# Bias Detection & Fairness Analysis

This notebook implements bias detection and fairness analysis for the NovaCred credit application dataset.
As part of the Data Governance Task Force, the goal is to identify potential discrimination in historical
lending decisions and quantify bias using established fairness metrics.

**Analyses performed:**
1. Gender Bias — Disparate Impact Ratio (four-fifths rule)
2. Statistical Significance Testing — Chi-squared test
3. Fairlearn Fairness Metrics — Demographic parity difference
4. Rejection Reason Analysis — Gender-based rejection patterns
5. Proxy Discrimination — Spending categories and income as gender proxies
6. Intersectional Analysis — Approval patterns across income brackets and gender

## 1. Setup & Data Loading

We load the consolidated dataset produced by the Data Engineering pipeline. This dataset
merges the original application data with pivoted spending behavior into a single file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from fairlearn.metrics import (
    demographic_parity_difference,
    demographic_parity_ratio,
    MetricFrame,
    selection_rate
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

In [ ]:
# Load consolidated dataset (spending behavior merged into main applications)
df = pd.read_csv('../data/processed/cleaned_credit_applications.csv')

print(f'Dataset shape: {df.shape[0]} records x {df.shape[1]} columns')
print(f'\nColumns:')
for i, col in enumerate(df.columns):
    print(f'  {i+1:2d}. {col}')
print(f'\nGender distribution:')
print(df['applicant_info.gender'].value_counts(dropna=False))
print(f'\nOverall approval rate: {df["decision.loan_approved"].mean():.1%}')

In [ ]:
# Filter to Male/Female for binary fairness analysis
# 2 records with 'Unknown' gender are excluded
df_bias = df[df['applicant_info.gender'].isin(['Male', 'Female'])].copy()
df_bias['gender_binary'] = (df_bias['applicant_info.gender'] == 'Male').astype(int)
df_bias['approved_binary'] = df_bias['decision.loan_approved'].astype(int)

print(f'Records for bias analysis: {len(df_bias)} (excluded {len(df) - len(df_bias)} with Unknown gender)')
print(f'Female: {len(df_bias[df_bias["applicant_info.gender"]=="Female"])}')
print(f'Male:   {len(df_bias[df_bias["applicant_info.gender"]=="Male"])}')

---
## 2. Disparate Impact Ratio (DIR) — Four-Fifths Rule

The **Disparate Impact Ratio** measures whether a selection process disproportionately affects a protected group.
Under the EEOC's **four-fifths (80%) rule**, a selection rate for any group less than 80% of the highest-performing
group constitutes evidence of adverse impact.

$$DIR = \frac{\text{Approval Rate}_{\text{unprivileged}}}{\text{Approval Rate}_{\text{privileged}}}$$

- **DIR < 0.8** → Potential disparate impact (violates four-fifths rule)
- **DIR = 1.0** → Perfect parity

In [ ]:
# Calculate approval rates by gender
approval_rates = df_bias.groupby('applicant_info.gender')['decision.loan_approved'].agg(['mean', 'sum', 'count'])
approval_rates.columns = ['Approval Rate', 'Approved Count', 'Total Count']
approval_rates['Rejected Count'] = approval_rates['Total Count'] - approval_rates['Approved Count']

print('=== Approval Statistics by Gender ===')
print(approval_rates.to_string())

# Compute Disparate Impact Ratio
rate_female = approval_rates.loc['Female', 'Approval Rate']
rate_male = approval_rates.loc['Male', 'Approval Rate']
DIR = rate_female / rate_male

print(f'\n{"="*50}')
print(f'Female Approval Rate: {rate_female:.1%} ({int(approval_rates.loc["Female","Approved Count"])}/{int(approval_rates.loc["Female","Total Count"])})')
print(f'Male Approval Rate:   {rate_male:.1%} ({int(approval_rates.loc["Male","Approved Count"])}/{int(approval_rates.loc["Male","Total Count"])})')
print(f'\nDisparate Impact Ratio (DIR): {DIR:.4f}')
print(f'Four-Fifths Threshold:         0.8000')
print(f'\nVERDICT: {"DISPARATE IMPACT DETECTED" if DIR < 0.8 else "No disparate impact"}')
print(f'The female approval rate is {DIR:.1%} of the male rate, which is')
print(f'{"BELOW" if DIR < 0.8 else "above"} the 80% threshold.')

In [ ]:
# Visualize approval rates with DIR threshold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of approval rates
colors = ['#e74c3c', '#3498db']
bars = axes[0].bar(['Female', 'Male'], [rate_female, rate_male], color=colors, edgecolor='black', linewidth=0.8)
axes[0].axhline(y=rate_male * 0.8, color='red', linestyle='--', linewidth=2, label=f'80% of Male rate ({rate_male*0.8:.1%})')
axes[0].set_ylabel('Approval Rate', fontsize=12)
axes[0].set_title('Loan Approval Rate by Gender', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=10)
for bar, rate in zip(bars, [rate_female, rate_male]):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{rate:.1%}', ha='center', va='bottom', fontsize=14, fontweight='bold')

# Stacked bar chart: approved vs denied
approval_data = approval_rates[['Approved Count', 'Rejected Count']]
approval_data.plot(kind='bar', stacked=True, ax=axes[1], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[1].set_title('Approved vs Denied by Gender', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Applications')
axes[1].set_xlabel('Gender')
axes[1].legend(['Approved', 'Denied'])
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../reports/gender_approval_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nDisparate Impact Ratio = {DIR:.4f} (threshold: 0.8)')

---
## 3. Statistical Significance Testing

We perform a **Chi-squared test of independence** to determine whether the relationship between
gender and loan approval is statistically significant (not due to chance).

- **H₀ (Null):** Gender and loan approval are independent
- **H₁ (Alternative):** Gender and loan approval are associated
- **Significance level:** α = 0.05

In [ ]:
# Contingency table
contingency = pd.crosstab(
    df_bias['applicant_info.gender'],
    df_bias['decision.loan_approved'],
    margins=True
)
contingency.columns = ['Rejected', 'Approved', 'Total']
contingency.index = ['Female', 'Male', 'Total']
print('=== Contingency Table ===')
print(contingency)

# Chi-squared test
ct = pd.crosstab(df_bias['applicant_info.gender'], df_bias['decision.loan_approved'])
chi2, p_value, dof, expected = stats.chi2_contingency(ct)

print(f'\n=== Chi-Squared Test of Independence ===')
print(f'Chi² statistic:     {chi2:.4f}')
print(f'p-value:            {p_value:.6f}')
print(f'Degrees of freedom: {dof}')

expected_df = pd.DataFrame(expected, index=['Female', 'Male'], columns=['Rejected', 'Approved'])
print(f'\nExpected frequencies (if independent):')
print(expected_df.round(1))

print(f'\n{"="*50}')
print(f'RESULT: The association is {"STATISTICALLY SIGNIFICANT" if p_value < 0.05 else "not significant"} (p = {p_value:.6f})')
if p_value < 0.05:
    print(f'We reject H₀: gender and loan approval are NOT independent.')

# Effect size: Cramér's V
n = len(df_bias)
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
effect = 'negligible' if cramers_v < 0.1 else 'small-to-moderate' if cramers_v < 0.3 else 'moderate' if cramers_v < 0.5 else 'large'
print(f'\nEffect Size (Cramér\'s V): {cramers_v:.4f} ({effect})')

In [ ]:
# Visualize observed vs expected
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(2)
width = 0.35

observed_female = [ct.loc['Female', False], ct.loc['Female', True]]
observed_male = [ct.loc['Male', False], ct.loc['Male', True]]

bars1 = ax.bar(x - width/2, observed_female, width, label='Female (Observed)', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, observed_male, width, label='Male (Observed)', color='#3498db', alpha=0.8)

# Expected values as dashed outlines
ax.bar(x - width/2, [expected_df.loc['Female', 'Rejected'], expected_df.loc['Female', 'Approved']],
       width, label='Female (Expected)', fill=False, edgecolor='#e74c3c', linewidth=2, linestyle='--')
ax.bar(x + width/2, [expected_df.loc['Male', 'Rejected'], expected_df.loc['Male', 'Approved']],
       width, label='Male (Expected)', fill=False, edgecolor='#3498db', linewidth=2, linestyle='--')

ax.set_xticks(x)
ax.set_xticklabels(['Rejected', 'Approved'], fontsize=12)
ax.set_ylabel('Number of Applicants', fontsize=12)
ax.set_title(f'Observed vs. Expected Counts by Gender\n(Chi² = {chi2:.2f}, p = {p_value:.4f})', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../reports/chi_squared_test.png', dpi=150, bbox_inches='tight')
plt.show()